# Ce script sert à trouver de nouveaux infinitifs propres à chaque locuteurs

##  <span style="color: #FFFFE0; font-weight: bold;">Listage de potentiels infinitifs avec glose</span>

In [3]:
import os

# Définir les chemins
terminaisons_path = "C:/Users/louis/OneDrive - Université de Poitiers/Fac/Master/Mémoire/Variation flexions/Localiser locuteurs/Terminaisons"
textes_poitevin_path = "C:/Users/louis/OneDrive - Université de Poitiers/Fac/Master/Mémoire/Variation flexions/Textes locuteurs/Poitevin/Textes en txt"
textes_saintongeais_path = "C:/Users/louis/OneDrive - Université de Poitiers/Fac/Master/Mémoire/Variation flexions/Textes locuteurs/Saintongeais/Textes en txt"
output_path_poitevin = "C:/Users/louis/OneDrive - Université de Poitiers/Fac/Master/Mémoire/Variation flexions/Textes locuteurs/Poitevin/Lexique locuteur/Correspondances"
output_path_saintongeais = "C:/Users/louis/OneDrive - Université de Poitiers/Fac/Master/Mémoire/Variation flexions/Textes locuteurs/Saintongeais/Lexique locuteur/Correspondances"
reference_path = "C:/Users/louis/OneDrive - Université de Poitiers/Fac/Master/Mémoire/Variation flexions/Textes locuteurs/Poitevin/Formes trouvées/Correspondances"

# Fonction pour extraire les terminaisons de l'infinitif d'un fichier
def extract_infinitive_endings(file_path):
    endings = set()
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                if line.startswith("* 1inf : "):
                    new_endings = line.split(":")[1].strip().split()
                    endings.update(new_endings)
    except Exception as e:
        print(f"Erreur lors de la lecture du fichier {file_path}: {e}")
    return list(endings)

# Fonction pour extraire les mots se terminant par des terminaisons spécifiques d'un texte
# On ne garde le cas que si le mot est immédiatement suivi de :, , ou =,
# puis on prend les 3 mots après la ponctuation.
def extract_words_with_context(text_path, endings):
    words_with_context = {}
    punctuations = {":", ",", "="}
    try:
        with open(text_path, 'r', encoding='latin1') as file:
            text = file.read().split()
            for i, word in enumerate(text):
                # Vérifier que le mot se termine par une des terminaisons (infinitif présumé)
                if len(word) >= 4 and any(word.lower().endswith(ending) for ending in endings):
                    # Vérifier qu'il y a un token juste après, et que c'est une ponctuation voulue
                    if i + 1 < len(text) and text[i + 1] in punctuations:
                        # Prendre les 3 mots suivant la ponctuation
                        following_words = ' '.join(text[i+2:i+5])
                        words_with_context[word.lower()] = following_words
                    # Sinon, on ne garde pas ce mot
    except Exception as e:
        print(f"Erreur lors de la lecture du fichier {text_path}: {e}")
    return words_with_context

# Fonction pour lire les mots de la colonne "Auteur" d'un fichier de référence
def read_reference_words(file_path):
    words = set()
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                word = line.split('\t')[0].strip().lower()
                if word:
                    words.add(word)
    except Exception as e:
        print(f"Erreur lors de la lecture du fichier de référence {file_path}: {e}")
    return words

# Traiter chaque auteur
authors = sorted(os.listdir(terminaisons_path))

for author in authors:
    terminaisons_file = os.path.join(terminaisons_path, author)
    endings = extract_infinitive_endings(terminaisons_file)

    # Fichier de référence pour cet auteur
    reference_file = os.path.join(reference_path, f"{os.path.splitext(author)[0]}_O.txt")
    reference_words = read_reference_words(reference_file) if os.path.exists(reference_file) else set()

    # Récupérer les fichiers de textes (Poitevin et Saintongeais) pour cet auteur
    author_prefix = os.path.splitext(author)[0]
    author_text_files_poitevin = [
        f for f in os.listdir(textes_poitevin_path)
        if f.startswith(author_prefix)
    ]
    author_text_files_saintongeais = [
        f for f in os.listdir(textes_saintongeais_path)
        if f.startswith(author_prefix)
    ]

    # Extraction des mots + contexte dans tous les textes de l'auteur
    all_words_with_context = {}

    for text_file in author_text_files_poitevin:
        text_file_path = os.path.join(textes_poitevin_path, text_file)
        words_with_context = extract_words_with_context(text_file_path, endings)
        all_words_with_context.update(words_with_context)

    for text_file in author_text_files_saintongeais:
        text_file_path = os.path.join(textes_saintongeais_path, text_file)
        words_with_context = extract_words_with_context(text_file_path, endings)
        all_words_with_context.update(words_with_context)

    # Choix du dossier de sortie (Poitevin ou Saintongeais)
    if author_text_files_saintongeais:
        output_file_path = os.path.join(
            output_path_saintongeais,
            f"{author_prefix}.txt"
        )
    else:
        output_file_path = os.path.join(
            output_path_poitevin,
            f"{author_prefix}.txt"
        )

    # Ne garder que les mots qui ne sont pas dans le fichier de référence
    cleaned_words_with_context = {
        word: context
        for word, context in all_words_with_context.items()
        if word not in reference_words
    }

    # Écrire directement les correspondances épurées dans le fichier unique
    try:
        with open(output_file_path, 'w', encoding='latin1') as output_file:
            for word in sorted(cleaned_words_with_context.keys()):
                output_file.write(f"{word} {cleaned_words_with_context[word]}\n")
    except Exception as e:
        print(f"Erreur lors de l'écriture du fichier {output_file_path}: {e}")

print("Fini :)")


Fini :)
